# SciFact Benchmark on Colab

This notebook runs the ScholarRAG SciFact bi-encoder benchmark on Google Colab with per-model isolation, live logs, and incremental result collection.

It evaluates `query -> abstract` retrieval over the SciFact test set: `300` unique queries over `5,183` abstracts.

## L4 inference setup

This notebook defaults to an `L4` GPU profile because we are benchmarking inference, not training.

The benchmark now mixes small and large bi-encoders so you can compare retrieval quality against latency:

- `BGE-EN-ICL` plain
- `BGE-EN-ICL` with few-shot examples
- `Qwen3-Embedding-0.6B`
- `Qwen3-Embedding-4B`
- `Qwen3-Embedding-8B`
- `Harrier OSS 0.6B`
- `EmbeddingGemma 300M`
- `SPECTER2`

`EmbeddingGemma 300M` is gated on Hugging Face, so this notebook reads `HF_TOKEN` from the environment or a one-cell override below.

The result files include both quality metrics and latency metrics, plus a direct `nDCG@10` vs latency tradeoff plot, so you can choose a practical quality/latency balance before moving on to reranking.

In [ ]:
# Optional: clone the repo only if you want a fresh remote checkout.
# Do not run this if you already uploaded or mounted an updated local copy.
# !git clone https://github.com/Wasiq-Malik/ScholarRAG.git /content/ScholarRAG


In [ ]:
%cd /content/ScholarRAG

!python -m pip install -U pip
!pip install -e .


In [ ]:
!nvidia-smi || true


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import os

# Optional: paste a Hugging Face token for gated models in this session only.
# Leave blank if HF_TOKEN is already set in Colab secrets or the environment.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    print("HF_TOKEN is not set. embeddinggemma_300m will fail until you provide it.")

RUN_NAME = f"colab_scifact_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = Path("benchmarks/scifact/results") / RUN_NAME

MODELS = [
    "bge_en_icl_plain",
    "bge_en_icl_examples",
    "qwen3_0_6b",
    "qwen3_4b",
    "qwen3_8b",
    "harrier_0_6b",
    "embeddinggemma_300m",
    "specter2",
]

DOCUMENT_MODE = "abstract"
SPLIT = "test"
TOP_K = 100
MAX_QUERIES = None
DEVICE = "cuda"
DTYPE = "auto"

GPU_PRESETS = {
    "l4_default": {
        "bge_en_icl_plain": 2,
        "bge_en_icl_examples": 2,
        "qwen3_0_6b": 16,
        "qwen3_4b": 4,
        "qwen3_8b": 2,
        "harrier_0_6b": 32,
        "embeddinggemma_300m": 64,
        "specter2": 64,
    },
    "l4_conservative": {
        "bge_en_icl_plain": 1,
        "bge_en_icl_examples": 1,
        "qwen3_0_6b": 8,
        "qwen3_4b": 2,
        "qwen3_8b": 1,
        "harrier_0_6b": 16,
        "embeddinggemma_300m": 32,
        "specter2": 32,
    },
}

GPU_PRESET = "l4_default"
BATCH_SIZE_OVERRIDES = dict(GPU_PRESETS[GPU_PRESET])

# Optional fine-grained overrides after loading the preset.
# Example: BATCH_SIZE_OVERRIDES['qwen3_8b'] = 1

print(json.dumps({
    "output_dir": str(OUTPUT_DIR),
    "models": MODELS,
    "split": SPLIT,
    "document_mode": DOCUMENT_MODE,
    "device": DEVICE,
    "dtype": DTYPE,
    "top_k": TOP_K,
    "max_queries": MAX_QUERIES,
    "gpu_preset": GPU_PRESET,
    "batch_size_overrides": BATCH_SIZE_OVERRIDES,
    "hf_token_set": bool(os.environ.get("HF_TOKEN")),
}, indent=2))


In [ ]:
from benchmarks.scifact.models import available_model_keys

registered_models = available_model_keys()
print("Registered benchmark models:", registered_models)
missing = sorted(set(MODELS) - set(registered_models))
if missing:
    raise ValueError(f"Notebook model list is not supported by the checked-out code: {missing}")


## Preset guidance

- Start with `l4_default`.
- Use `l4_conservative` only if one of the larger models is unstable.
- If only one model needs adjustment, keep the main preset and reduce only that model in `BATCH_SIZE_OVERRIDES`.

In [ ]:
import json
import os
import shlex
import signal
import subprocess
import sys
from pathlib import Path

def run_one_model(model_key: str) -> dict:
    model_dir = OUTPUT_DIR / model_key
    model_dir.mkdir(parents=True, exist_ok=True)
    log_path = model_dir / "run.log"

    cmd = [
        sys.executable,
        "benchmarks/scifact/run_benchmark.py",
        "--split", SPLIT,
        "--document-mode", DOCUMENT_MODE,
        "--device", DEVICE,
        "--dtype", DTYPE,
        "--top-k", str(TOP_K),
        "--output-dir", str(model_dir),
        "--models", model_key,
        "--show-progress",
    ]

    if MAX_QUERIES is not None:
        cmd += ["--max-queries", str(MAX_QUERIES)]

    batch_size = BATCH_SIZE_OVERRIDES.get(model_key)
    if batch_size is not None:
        cmd += ["--batch-size", str(batch_size)]

    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"

    print(f"\n=== Running {model_key} ===")
    print("Command:")
    print(" ".join(shlex.quote(part) for part in cmd))
    print(f"Log file: {log_path}")
    print(f"Configured batch size: {batch_size}")

    with log_path.open("w") as log_handle:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_handle.write(line)
        returncode = process.wait()

    result = {
        "model": model_key,
        "batch_size": batch_size,
        "returncode": returncode,
        "status": "completed" if returncode == 0 else "failed",
        "log_path": str(log_path),
        "output_dir": str(model_dir),
    }

    if returncode == -signal.SIGKILL:
        result["failure_reason"] = "likely_oom_or_os_kill"
    elif returncode != 0:
        result["failure_reason"] = f"nonzero_exit_{returncode}"

    status_path = model_dir / "notebook_status.json"
    status_path.write_text(json.dumps(result, indent=2))
    return result


all_results = []
for model_key in MODELS:
    all_results.append(run_one_model(model_key))

aggregate_status_path = OUTPUT_DIR / "notebook_status.json"
aggregate_status_path.parent.mkdir(parents=True, exist_ok=True)
aggregate_status_path.write_text(json.dumps(all_results, indent=2))
all_results

In [ ]:
import pandas as pd
from benchmarks.scifact.generate_report import build_report

summary_rows = []
for result in all_results:
    model_dir = Path(result["output_dir"])
    summary_path = model_dir / "summary.csv"
    if summary_path.exists():
        frame = pd.read_csv(summary_path)
        if not frame.empty:
            summary_rows.append(frame)

if summary_rows:
    aggregate_summary = pd.concat(summary_rows, ignore_index=True)
    aggregate_summary = aggregate_summary.sort_values("ndcg_at_10", ascending=False)
    aggregate_summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
    (OUTPUT_DIR / "summary.json").write_text(aggregate_summary.to_json(orient="records", indent=2))

    aggregate_config = {
        "split": SPLIT,
        "document_mode": DOCUMENT_MODE,
        "device": DEVICE,
        "dtype": DTYPE,
        "models": MODELS,
        "query_count": int(aggregate_summary.iloc[0]["query_count"]),
        "corpus_size": int(aggregate_summary.iloc[0]["corpus_size"]),
        "top_k": TOP_K,
        "bge_use_examples": BGE_USE_EXAMPLES,
        "gpu_preset": GPU_PRESET,
        "batch_size_overrides": BATCH_SIZE_OVERRIDES,
    }
    (OUTPUT_DIR / "config.json").write_text(json.dumps(aggregate_config, indent=2))
    report_path = build_report(OUTPUT_DIR)
    print(f"Wrote aggregate report to {report_path}")
    aggregate_summary
else:
    print("No successful model runs produced a summary.csv file.")

In [ ]:
import pandas as pd

pd.DataFrame(all_results)

In [ ]:
from IPython.display import Markdown, display

report_path = OUTPUT_DIR / "REPORT.md"
if report_path.exists():
    display(Markdown(report_path.read_text()))
else:
    print("No aggregate report found yet.")

In [ ]:
from IPython.display import Image, display

plots_dir = OUTPUT_DIR / "plots"
for name in ["ndcg_at_10.png", "mrr_at_10.png", "recall_at_100.png", "runtime_seconds.png", "avg_query_end_to_end_latency_ms.png", "ndcg_vs_avg_query_latency.png"]:
    path = plots_dir / name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))

In [ ]:
# Inspect a specific model log after a failure.
MODEL_TO_INSPECT = MODELS[5]
log_path = OUTPUT_DIR / MODEL_TO_INSPECT / "run.log"
if log_path.exists():
    print(log_path)
    print(log_path.read_text()[-12000:])
else:
    print(f"No log found for {MODEL_TO_INSPECT}")

In [ ]:
# Rerun a single model with a lower batch size if needed.
# Example:
# BATCH_SIZE_OVERRIDES['qwen3_8b'] = 1
# run_one_model('qwen3_8b')

In [ ]:
# Optional: zip the aggregate directory for download from Colab.
!zip -r "{OUTPUT_DIR}.zip" "{OUTPUT_DIR}"